# Train YOLOv8n cho Drone trên Google Colab

Hướng dẫn từng bước để train mô hình YOLOv8n (nano) trên Google Colab, sau đó xuất ra định dạng để chạy trên drone.

**Bước đầu tiên:** vào menu `Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save`, rồi chạy lần lượt từng ô từ trên xuống.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print(">>> Chưa có GPU. Vào menu: Runtime > Change runtime type > chọn 'T4 GPU' rồi chạy lại.")

In [ ]:
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Tải dataset từ Roboflow

Dataset đã được export sẵn sang định dạng YOLOv8, kèm file `data.yaml` (chứa tên class). Ô dưới đây sẽ tải về và giải nén vào `/content/`.

In [ ]:
# Tải dataset từ Roboflow — KHÔNG hardcode key (tránh lộ trên GitHub).
# Key đọc theo thứ tự: Colab Secrets (userdata) -> env var -> file .env
# Trên Colab: nút 🔑 "Secrets" bên trái -> + New secret: ROBOFLOW_API_KEY.

def _get_roboflow_key():
    import os
    # 1) Colab Secrets (chạy trên Colab)
    try:
        from google.colab import userdata
        return userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        pass
    # 2) Biến môi trường
    if os.environ.get("ROBOFLOW_API_KEY"):
        return os.environ["ROBOFLOW_API_KEY"]
    # 3) File .env (chạy local)
    try:
        for line in open(".env"):
            line = line.strip()
            if line.startswith("ROBOFLOW_API_KEY") and "=" in line:
                return line.split("=", 1)[1].strip()
    except FileNotFoundError:
        pass
    raise ValueError(
        "Thiếu ROBOFLOW_API_KEY. Trên Colab: nút 🔑 Secrets bên trái -> "
        "+ New secret, tên ROBOFLOW_API_KEY. Local thì tạo file .env."
    )

ROBOFLOW_API_KEY = _get_roboflow_key()
print("Đã lấy ROBOFLOW_API_KEY")

from roboflow import Roboflow

WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
project.version(PROJECT_VERSION).download("yolov8")
print("Dataset đã tải về /content/")

In [ ]:
# Tự động tìm thư mục dataset vừa tải (thư mục chứa data.yaml)
import os, glob

candidates = glob.glob("/content/*/data.yaml")
if candidates:
    DATASET_PATH = os.path.dirname(candidates[0])
else:
    # Nếu không tìm thấy, sửa lại đường dẫn tên thư mục cho đúng
    DATASET_PATH = "/content/citrus-disease-detection-1"

print("DATASET_PATH =", DATASET_PATH)
print("Cấu trúc thư mục:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    print("  " * level + "|" + os.path.basename(root) + "/")
    for f in files[:3]:
        print("  " * (level + 1) + f)

import yaml
with open(os.path.join(DATASET_PATH, "data.yaml")) as f:
    cfg = yaml.safe_load(f)
print("\nSố class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load model & train

File `yolov8n.pt` (mô hình nano) sẽ được tải tự động lần đầu tiên.

In [ ]:
import os

if os.path.exists("/content/yolov8n.pt"):
    model_path = "/content/yolov8n.pt"
elif os.path.exists("/content/drive/MyDrive/yolov8n.pt"):
    model_path = "/content/drive/MyDrive/yolov8n.pt"
else:
    model_path = "yolov8n.pt"  # ultralytics tự tải về

print("Dùng model:", model_path)
model = YOLO(model_path)

In [ ]:
# ===== Cấu hình train (best practice) =====
# epochs cao + patience tự dừng: yolov8n trên 16k ảnh cần ~100+ epochs mới hội tụ.
# Bằng chứng: run 100 ep có đỉnh mAP50 ở epoch 57 và mAP50-95 ở 70.
#   -> 50 ep là thiếu (v1 mAP50 0.531 vs v2 0.554).
#   -> time=8 giới hạn cứng: tránh vượt session Kaggle 9-12h.
EPOCHS = 100      # mục tiêu; patience sẽ dừng sớm khi không còn cải thiện
IMGSZ = 640       # khớp với export & kmodel
BATCH = 16
PATIENCE = 15

RESULTS_DIR = "/content/runs/drone_yolov8n/weights"
OUT_DIR     = "/content/drive/MyDrive/drone_yolo"
os.makedirs(OUT_DIR, exist_ok=True)

def _backup_to_drive(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        dst = os.path.join(OUT_DIR, "best_checkpoint.pt")
        shutil.copy(src, dst)
        print(f"  [backup epoch {trainer.epoch}] best.pt -> Drive", flush=True)
    except Exception as e:
        print("  [backup fail]", e, flush=True)

# ===== BACKUP mỗi epoch: copy best.pt vào thư mục Output để tải được bất kỳ lúc nào =====
from ultralytics.utils import callbacks


callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

# ===== TĂNG RECALL (model citrus dễ bỏ sót bệnh: P cao / R thấp) =====
# - fliplr=0.5 + scale=0.5: đa dạng vị trí/kích thước lá
# KHÔNG dùng flipud/degrees/shear/perspective mạnh vì ảnh drone trên cao
# và lá bệnh có hình dạng có ý nghĩa — biến dạng quá làm sai nhãn.
print(f"Train yolov8n: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=EPOCHS,
    time=5,               # tối đa 5 giờ — T4 miễn phí hay bị thu hồi
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    fliplr=0.5,           # lật ngang
    scale=0.5,            # zoom ngẫu nhiên -> lá to nhỏ khác nhau
    cache=True,
    workers=2,
    cos_lr=True,          # giảm learning rate theo cos, hội tụ tốt hơn
    project="/content/runs",
    name="drone_yolov8n",
)


## 3. Đánh giá kết quả

In [ ]:
# Đánh giá — in cả recall (quan trọng: model hay bỏ sót bệnh)
metrics = model.val()
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
import glob

# Dự đoán thử trên ảnh test (Roboflow dùng thư mục valid/test thay vì val)
test_imgs = (glob.glob(f"{DATASET_PATH}/test/images/*.*")
             or glob.glob(f"{DATASET_PATH}/valid/images/*.*")
             or glob.glob(f"{DATASET_PATH}/val/images/*.*"))
print("Tìm thấy", len(test_imgs), "ảnh")

if test_imgs:
    # conf hạ xuống 0.15 để xem model bắt được nhiều bệnh hơn (ưu tiên recall)
    model.predict(source=test_imgs[:8], conf=0.15, save=True)
    # Ảnh kết quả nằm trong thư mục runs/detect/predict
    from IPython.display import Image
    display(Image(filename="/content/runs/detect/predict/" + os.path.basename(test_imgs[0])))

## 4. Xuất model để chạy trên drone

Chọn định dạng phù hợp với phần cứng bạn dùng để bay (điều khiển trên bo mạch):
- **Jetson Nano / Jetson Orin (NVIDIA)**: `tensorrt` hoặc `onnx`
- **Raspberry Pi / board nhúng**: `ncnn` hoặc `onnx`
- **Điện thoại Android / Edge TPU**: `tflite`
- **Máy tính Intel**: `openvino`

In [ ]:
# ===== EXPORT: 1 lần train -> 2 phiên bản =====
# [LAPTOP test]  best.pt   -> chạy trực tiếp bằng ultralytics trên máy bạn
# [DRONE K230]   best.onnx -> qua nncase -> best.kmodel để chạy trên board

# (1) ONNX dành cho K230 — KHÔNG dùng half=True để nncase nạp được
#     imgsz=IMGSZ phải KHỚP với model_input_size trong k230_yolov8_det.py
model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)

# (2) Bản int8 lượng tử, nhẹ hơn (tùy chọn)
model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True, int8=True, data=f"{DATASET_PATH}/data.yaml")

print("Export xong. File trong:", RESULTS_DIR)
print("  - best.pt   -> LAPTOP (chạy realtime bằng ultralytics)")
print("  - best.onnx -> DRONE K230 (convert_to_kmodel.py -> best.kmodel)")

In [ ]:
# Lưu mọi phiên bản model về Google Drive để không mất khi Colab đóng session.
import shutil, glob

SAVE_DIR = "/content/drive/MyDrive/drone_yolo"
os.makedirs(SAVE_DIR, exist_ok=True)

# best.pt + last.pt + mọi .onnx / .tflite vừa export
files = glob.glob(f"{RESULTS_DIR}/best.*") + glob.glob(f"{RESULTS_DIR}/last.*")

for src in files:
    shutil.copy(src, os.path.join(SAVE_DIR, os.path.basename(src)))
    print("Đã lưu:", os.path.basename(src))

print("Thư mục Drive:", SAVE_DIR)

## Mẹo khi train cho drone

- **Vật thể nhỏ**: ảnh chụp từ drone thường có vật thể rất nhỏ. Thử tăng `imgsz` lên 800 hoặc dùng kỹ thuật tile/split ảnh.
- **Không lật dọc (flipud)**: với ảnh trên cao, giữ nguyên default `flipud=0.0` để tránh vật thể lộn ngược không đúng thực tế. `fliplr=0.5` (lật ngang) vẫn nên giữ.
- **Augmentation**: `mosaic` mặc định là 1.0, rất hữu ích cho vật thể nhỏ.
- **Dừng sớm**: nếu thấy `best.pt` không cải thiện ở nhiều epoch, giảm `patience` để đỡ tốn thời gian.
- **Session mất kết nối**: Colab miễn phí có thể ngắt sau vài giờ. Nên đã mount Drive từ đầu để `best.pt` được lưu lại, hoặc dùng Colab Pro để train lâu hơn.